In [27]:
import pandas as pd
import json
from enum import Enum
from typing import Dict, List, Set, Any, Optional

# ==========================================
# 1. ENUMS & CONFIGURATION (No Hard Coding)
# ==========================================

class FilePaths(str, Enum):
    """File paths for input and output data."""
    ACCOUNTS_JSON_PATH = "sfa.accounts-Ayush.json"
    CALLS_JSON_PATH = "sfa.accountCalls-Ayush.json"
    OUTPUT_EXCEL_PATH = "mapped_sales_report.xlsx"
    FALLBACK_RAW_CSV = "mapped_sales_raw_data.csv"
    FALLBACK_SUMMARY_CSV = "account_type_summary.csv"

class ColumnNames(str, Enum):
    """Expected column names in the final DataFrame."""
    ACCOUNT_ID = "Account ID"
    ACCOUNT_NAME = "Account Name"
    ERP_CODE = "ERP Code"
    ACCOUNT_TYPES = "Account Types"
    ZONE = "Zone"
    REGION = "Region"
    AREA = "Area"
    BUSINESS_UNIT = "Business Unit"
    TERRITORY = "Territory"
    BRANDS_ORDERED = "Brands Ordered"
    SKU_CODES = "SKU Codes"

class JsonKeys(str, Enum):
    """Keys used to traverse the MongoDB JSON documents."""
    ID_OBJECT = "_id"
    OID = "$oid"
    NAME = "name"
    ERP_CODE = "erpCode"
    LOCATIONS = "locations"
    LOCATION_TYPE = "locationType"
    LOCATION_HIERARCHY = "locationHierarchy"
    ZONE = "Zone"
    REGION = "Region"
    AREA = "Area"
    BUSINESS_UNIT = "businessUnit"
    TERRITORY_IDENTIFIER = "Territory"

    # Call Document Specific Keys
    CALL_ACCOUNT = "account"
    ACCOUNT_TYPE = "accountType"
    SUPPLIER = "supplier"
    ACCOUNT_ID_REF = "accountId"
    ORDER_LINE_ITEMS = "orderLineItems"
    NUTRITION_ORDER_ITEMS = "nutritionOrderLineItems"
    BRAND = "brand"
    SKU_CODE = "skuCode"

In [28]:
# ==========================================
# 2. DATA EXTRACTION MODULES
# ==========================================

def extract_mongo_object_id(id_container: Any) -> str:
    """Extracts the string ID from a MongoDB ObjectId dictionary."""
    if not isinstance(id_container, dict):
        return ""
    return str(id_container.get(JsonKeys.OID.value, ""))

def extract_location_hierarchy(locations_list: Any) -> Dict[str, str]:
    """
    Iterates through the locations array to find the 'Territory' object
    and extracts its hierarchy details.
    """
    hierarchy_data = {
        ColumnNames.ZONE.value: "",
        ColumnNames.REGION.value: "",
        ColumnNames.AREA.value: "",
        ColumnNames.BUSINESS_UNIT.value: "",
        ColumnNames.TERRITORY.value: ""
    }

    if not isinstance(locations_list, list):
        return hierarchy_data

    for location_element in locations_list:
        if location_element.get(JsonKeys.LOCATION_TYPE.value) == JsonKeys.TERRITORY_IDENTIFIER.value:
            hierarchy = location_element.get(JsonKeys.LOCATION_HIERARCHY.value, {})

            hierarchy_data[ColumnNames.ZONE.value] = hierarchy.get(JsonKeys.ZONE.value, "")
            hierarchy_data[ColumnNames.REGION.value] = hierarchy.get(JsonKeys.REGION.value, "")
            hierarchy_data[ColumnNames.AREA.value] = hierarchy.get(JsonKeys.AREA.value, "")
            hierarchy_data[ColumnNames.BUSINESS_UNIT.value] = location_element.get(JsonKeys.BUSINESS_UNIT.value, "")
            hierarchy_data[ColumnNames.TERRITORY.value] = location_element.get(JsonKeys.NAME.value, "")
            break # Stop after finding the primary territory

    return hierarchy_data

def extract_order_line_items(
    items_list: Any,
    collected_brands: Set[str],
    collected_skus: Set[str]
) -> None:
    """
    Mutates the provided sets by adding unique brands and SKUs from a list of order items.
    """
    if not isinstance(items_list, list):
        return

    for item in items_list:
        brand = item.get(JsonKeys.BRAND.value)
        sku = item.get(JsonKeys.SKU_CODE.value)

        if brand: collected_brands.add(str(brand))
        if sku: collected_skus.add(str(sku))

In [29]:
# ==========================================
# 3. DOCUMENT PROCESSING MODULES
# ==========================================

def process_single_account_document(account_document: Dict[str, Any]) -> Dict[str, Any]:
    """Processes a single raw dictionary from the Accounts JSON."""
    account_id = extract_mongo_object_id(account_document.get(JsonKeys.ID_OBJECT.value, {}))
    account_name = account_document.get(JsonKeys.NAME.value, "")
    erp_code = account_document.get(JsonKeys.ERP_CODE.value, "")

    # Base location hierarchy pulled from Accounts file
    locations = account_document.get(JsonKeys.LOCATIONS.value, [])
    location_details = extract_location_hierarchy(locations)

    return {
        ColumnNames.ACCOUNT_ID.value: account_id,
        ColumnNames.ACCOUNT_NAME.value: account_name,
        ColumnNames.ERP_CODE.value: erp_code,
        ColumnNames.ZONE.value: location_details[ColumnNames.ZONE.value],
        ColumnNames.REGION.value: location_details[ColumnNames.REGION.value],
        ColumnNames.AREA.value: location_details[ColumnNames.AREA.value],
        ColumnNames.BUSINESS_UNIT.value: location_details[ColumnNames.BUSINESS_UNIT.value],
        ColumnNames.TERRITORY.value: location_details[ColumnNames.TERRITORY.value]
    }

def process_single_call_document(call_document: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """Processes a single raw dictionary from the Account Calls JSON."""
    if not call_document:
        return None

    # Link relation: Pull ID strictly from the Supplier node
    supplier_node = call_document.get(JsonKeys.SUPPLIER.value, {})
    account_id = extract_mongo_object_id(supplier_node.get(JsonKeys.ACCOUNT_ID_REF.value, {}))

    if not account_id:
        return None

    unique_account_types: Set[str] = set()
    unique_brands: Set[str] = set()
    unique_skus: Set[str] = set()

    # Extract accountType strictly from the main 'account' node in calls
    account_node = call_document.get(JsonKeys.CALL_ACCOUNT.value, {})
    call_account_type = account_node.get(JsonKeys.ACCOUNT_TYPE.value)

    if call_account_type:
        unique_account_types.add(str(call_account_type).strip())

    # Process standard and nutrition line items for Brands and SKUs
    standard_items = call_document.get(JsonKeys.ORDER_LINE_ITEMS.value, [])
    nutrition_items = call_document.get(JsonKeys.NUTRITION_ORDER_ITEMS.value, [])

    extract_order_line_items(standard_items, unique_brands, unique_skus)
    extract_order_line_items(nutrition_items, unique_brands, unique_skus)

    return {
        ColumnNames.ACCOUNT_ID.value: account_id,
        "Call_Account_Types": list(unique_account_types),
        ColumnNames.BRANDS_ORDERED.value: list(unique_brands),
        ColumnNames.SKU_CODES.value: list(unique_skus)
    }

def aggregate_call_data(raw_calls_list: List[Dict[str, Any]]) -> pd.DataFrame:
    """Aggregates multiple calls under the same Account ID to avoid duplicates."""
    aggregated_list = []

    for doc in raw_calls_list:
        processed_call = process_single_call_document(doc)
        if processed_call:
            aggregated_list.append(processed_call)

    processed_calls_df = pd.DataFrame(aggregated_list)

    if processed_calls_df.empty:
        return processed_calls_df

    # Group by Account ID and flatten nested lists into unique sets
    grouped_calls = processed_calls_df.groupby(ColumnNames.ACCOUNT_ID.value).agg({
        "Call_Account_Types": lambda types_lists: list(set([item for sublist in types_lists for item in sublist if item])),
        ColumnNames.BRANDS_ORDERED.value: lambda brands_lists: list(set([item for sublist in brands_lists for item in sublist if item])),
        ColumnNames.SKU_CODES.value: lambda skus_lists: list(set([item for sublist in skus_lists for item in sublist if item]))
    }).reset_index()

    return grouped_calls

In [30]:
# ==========================================
# 4. FINAL MERGE & REPORT GENERATION MODULES
# ==========================================

def merge_and_format_raw_dataset(accounts_df: pd.DataFrame, calls_df: pd.DataFrame) -> pd.DataFrame:
    """Merges the processed accounts and calls DataFrames to generate the exploded raw data."""

    # Inner join ensures only Accounts with a matching supplier.accountId in Calls are mapped
    merged_df = pd.merge(
        accounts_df,
        calls_df,
        on=ColumnNames.ACCOUNT_ID.value,
        how='inner'
    )

    # Explode lists into separate rows
    merged_df = merged_df.explode("Call_Account_Types")
    merged_df = merged_df.explode(ColumnNames.BRANDS_ORDERED.value)
    merged_df = merged_df.explode(ColumnNames.SKU_CODES.value)

    # --- CRITICAL FIX: Reset the index after exploding to avoid duplicate labels error ---
    merged_df.reset_index(drop=True, inplace=True)

    # Replace NaN with an empty string just in case the list was completely empty
    merged_df["Call_Account_Types"] = merged_df["Call_Account_Types"].fillna("")
    merged_df[ColumnNames.BRANDS_ORDERED.value] = merged_df[ColumnNames.BRANDS_ORDERED.value].fillna("")
    merged_df[ColumnNames.SKU_CODES.value] = merged_df[ColumnNames.SKU_CODES.value].fillna("")

    # Assign exploded variables to the final columns as strings
    merged_df[ColumnNames.ACCOUNT_TYPES.value] = merged_df["Call_Account_Types"].astype(str)
    merged_df[ColumnNames.BRANDS_ORDERED.value] = merged_df[ColumnNames.BRANDS_ORDERED.value].astype(str)
    merged_df[ColumnNames.SKU_CODES.value] = merged_df[ColumnNames.SKU_CODES.value].astype(str)

    # Filter to requested columns only
    final_columns = [
        ColumnNames.ACCOUNT_ID.value,
        ColumnNames.ACCOUNT_NAME.value,
        ColumnNames.ERP_CODE.value,
        ColumnNames.ACCOUNT_TYPES.value,
        ColumnNames.ZONE.value,
        ColumnNames.REGION.value,
        ColumnNames.AREA.value,
        ColumnNames.BUSINESS_UNIT.value,
        ColumnNames.TERRITORY.value,
        ColumnNames.BRANDS_ORDERED.value,
        ColumnNames.SKU_CODES.value
    ]

    return merged_df[final_columns]

def generate_summary_report(raw_data_df: pd.DataFrame) -> pd.DataFrame:
    """
    Generates a summary report counting unique Account IDs for each Account Type
    across the different Business Units (e.g., VACCINES DOMESTIC, FAN).
    """
    # 1. Isolate the columns needed and drop duplicates.
    # This prevents overcounting accounts that were exploded into multiple rows due to Brands/SKUs.
    base_summary_df = raw_data_df[[
        ColumnNames.ACCOUNT_ID.value,
        ColumnNames.ACCOUNT_TYPES.value,
        ColumnNames.BUSINESS_UNIT.value
    ]].drop_duplicates()

    # 2. Filter out rows where Account Type is empty (if any)
    base_summary_df = base_summary_df[base_summary_df[ColumnNames.ACCOUNT_TYPES.value] != ""]

    # 3. Create a crosstab pivot table counting the occurrences
    summary_pivot = pd.crosstab(
        index=base_summary_df[ColumnNames.ACCOUNT_TYPES.value],
        columns=base_summary_df[ColumnNames.BUSINESS_UNIT.value]
    )

    # 4. Clean up the DataFrame structure for export
    summary_pivot.reset_index(inplace=True)
    summary_pivot.rename_axis(None, axis=1, inplace=True)

    return summary_pivot

In [31]:
# ==========================================
# 5. FILE I/O MODULES
# ==========================================

def load_json_file(file_path: str) -> List[Dict[str, Any]]:
    """Loads a JSON file handling both standard JSON Arrays and NDJSON formats."""
    with open(file_path, 'r', encoding='utf-8') as f:
        try:
            return json.load(f)
        except json.JSONDecodeError:
            f.seek(0)
            return [json.loads(line) for line in f if line.strip()]

In [32]:
# ==========================================
# 6. MAIN EXECUTION (Definitive Path)
# ==========================================

def execute_data_mapping_pipeline() -> None:
    """Main orchestration function."""
    print("Initializing Data Mapping Pipeline...")

    try:
        # 1. Load Native JSON Data
        print("Loading JSON files...")
        raw_accounts_list = load_json_file(FilePaths.ACCOUNTS_JSON_PATH.value)
        raw_calls_list = load_json_file(FilePaths.CALLS_JSON_PATH.value)

        # 2. Process Data
        print("Processing Account Records...")
        processed_accounts_list = [process_single_account_document(doc) for doc in raw_accounts_list]
        accounts_df = pd.DataFrame(processed_accounts_list)

        print("Processing Call Records...")
        aggregated_calls_df = aggregate_call_data(raw_calls_list)

        # 3. Generate Reports
        print("Generating Raw Dataset...")
        raw_dataset_df = merge_and_format_raw_dataset(accounts_df, aggregated_calls_df)

        print("Generating Summary Report...")
        summary_dataset_df = generate_summary_report(raw_dataset_df)

        # 4. Export to Excel
        print("Exporting data...")
        try:
            with pd.ExcelWriter(FilePaths.OUTPUT_EXCEL_PATH.value, engine='openpyxl') as writer:
                raw_dataset_df.to_excel(writer, sheet_name="Raw Data", index=False)
                summary_dataset_df.to_excel(writer, sheet_name="Summary Report", index=False)
            print(f"Success! Data exported to multiple sheets in: {FilePaths.OUTPUT_EXCEL_PATH.value}")

        except ModuleNotFoundError:
            # Fallback for systems without openpyxl installed
            print("\nNOTE: 'openpyxl' module is missing. Generating two CSV files instead of an Excel workbook.")
            print("To generate Excel files in the future, run: pip install openpyxl\n")

            raw_dataset_df.to_csv(FilePaths.FALLBACK_RAW_CSV.value, index=False)
            summary_dataset_df.to_csv(FilePaths.FALLBACK_SUMMARY_CSV.value, index=False)

            print(f"Success! Exported to '{FilePaths.FALLBACK_RAW_CSV.value}' and '{FilePaths.FALLBACK_SUMMARY_CSV.value}'.")

    except FileNotFoundError as fnf_error:
        print(f"File Error: Ensure the JSON files are in the same directory. {fnf_error}")
    except Exception as execution_error:
        print(f"Pipeline failed due to an error: {execution_error}")

if __name__ == "__main__":
    execute_data_mapping_pipeline()

Initializing Data Mapping Pipeline...
Loading JSON files...
Processing Account Records...
Processing Call Records...
Generating Raw Dataset...
Generating Summary Report...
Exporting data...
Success! Data exported to multiple sheets in: mapped_sales_report.xlsx


In [15]:
import pandas as pd
import ast
import ijson

# ==========================================
# 1. FILE CONFIGURATION
# ==========================================
EXCEL_FILE = 'distributor list.xlsx'
SHEET_1 = '7N'
SHEET_2 = 'Vaccine'

CALLS_JSON_PATH = 'sfa.accountCalls-Full.json'

OUTPUT_MERGED_CSV = 'supplier_types_merged.csv'
OUTPUT_SEPARATED_CSV = 'supplier_types_separated.csv'

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def extract_supplier_data(excel_filepath: str, sheet_name: str) -> dict:
    """Reads the distributor Excel sheet and extracts all supplier details mapped by their ID."""
    supplier_info = {}
    try:
        df = pd.read_excel(excel_filepath, sheet_name=sheet_name, skiprows=1, engine='openpyxl')
        for index, row in df.iterrows():
            val = row.get('supplier.AccountId')
            if pd.isna(val):
                continue

            try:
                parsed_dict = ast.literal_eval(str(val))
                if isinstance(parsed_dict, dict) and '$oid' in parsed_dict:
                    supplier_id = parsed_dict['$oid']
                    supplier_info[supplier_id] = {
                        "Supplier Name": row.get('supplier.name', ''),
                        "ERP Code": row.get('supplier.erpCode', ''),
                        "Region": row.get('Region', ''),
                        "Area": row.get('Area', ''),
                        "Territory": row.get('Territory', '')
                    }
            except (ValueError, SyntaxError):
                continue
        return supplier_info
    except Exception as e:
        print(f"Failed to read sheet '{sheet_name}': {e}")
        return {}

def standardize_account_type(raw_type: str) -> str:
    """Standardizes specific account types into 'Institution'."""
    if not raw_type:
        return ""
    clean_type = str(raw_type).strip()
    normalized = clean_type.lower().replace(" ", "")
    if normalized in ['hospital', 'maternityhome', 'nursinghome']:
        return 'Institution'
    return clean_type

def classify_supplier_mix(account_types_str: str, detailed: bool = True) -> str:
    """
    Dynamically categorizes the supplier into 'Only' or 'Mix' categories.
    If detailed is False, it just returns 'Mix' instead of 'Mix (A & B)'.
    """
    if not account_types_str:
        return "No Data"

    types = account_types_str.split(", ")

    if len(types) == 1:
        return f"{types[0]} Only"
    else:
        if detailed:
            joined_mix = " & ".join(types)
            return f"Mix ({joined_mix})"
        else:
            return "Mix"

# ==========================================
# 3. MAIN EXECUTION
# ==========================================

def main():
    print(f"Loading Target Supplier Data from '{EXCEL_FILE}'...")

    data_7n = extract_supplier_data(EXCEL_FILE, SHEET_1)
    data_vaccine = extract_supplier_data(EXCEL_FILE, SHEET_2)

    target_suppliers_data = {**data_7n, **data_vaccine}
    print(f"Total Unique Supplier IDs extracted: {len(target_suppliers_data)}")

    if not target_suppliers_data:
        print("No IDs found. Exiting.")
        return

    supplier_mapping = {}

    print(f"\nScanning '{CALLS_JSON_PATH}' iteratively... (This may take a minute)")
    try:
        with open(CALLS_JSON_PATH, 'rb') as f:
            try:
                objects = ijson.items(f, 'item')
                for call in objects:
                    if not call or not isinstance(call, dict): continue
                    supplier_node = call.get('supplier') or {}
                    account_id_node = supplier_node.get('accountId') or {}
                    supplier_id = account_id_node.get('$oid', '') if isinstance(account_id_node, dict) else ''

                    if supplier_id in target_suppliers_data:
                        account_node = call.get('account') or {}
                        raw_type = account_node.get('accountType')
                        if raw_type:
                            standardized_type = standardize_account_type(raw_type)
                            if standardized_type:
                                if supplier_id not in supplier_mapping:
                                    supplier_mapping[supplier_id] = set()
                                supplier_mapping[supplier_id].add(standardized_type)

            except ijson.common.IncompleteJSONError:
                f.seek(0)
                objects = ijson.items(f, '', multiple_values=True)
                for call in objects:
                    if not call or not isinstance(call, dict): continue
                    supplier_node = call.get('supplier') or {}
                    account_id_node = supplier_node.get('accountId') or {}
                    supplier_id = account_id_node.get('$oid', '') if isinstance(account_id_node, dict) else ''

                    if supplier_id in target_suppliers_data:
                        account_node = call.get('account') or {}
                        raw_type = account_node.get('accountType')
                        if raw_type:
                            standardized_type = standardize_account_type(raw_type)
                            if standardized_type:
                                if supplier_id not in supplier_mapping:
                                    supplier_mapping[supplier_id] = set()
                                supplier_mapping[supplier_id].add(standardized_type)

    except FileNotFoundError:
        print(f"\nError: Could not find '{CALLS_JSON_PATH}'.")
        return
    except Exception as e:
        print(f"\nError during parsing: {e}")
        return

    # ==========================================
    # 4. DATAFRAME CREATION & EXPORT
    # ==========================================
    print("\nProcessing matched data...")

    data_rows = []
    for sid, type_set in supplier_mapping.items():
        merged_string = ", ".join(sorted(type_set))
        base_info = target_suppliers_data.get(sid, {})

        data_rows.append({
            "Supplier ID": sid,
            "Supplier Name": base_info.get("Supplier Name", ""),
            "ERP Code": base_info.get("ERP Code", ""),
            "Region": base_info.get("Region", ""),
            "Area": base_info.get("Area", ""),
            "Territory": base_info.get("Territory", ""),
            "Merged Account Types": merged_string
        })

    df_merged = pd.DataFrame(data_rows)

    if df_merged.empty:
        print("No matching records found in the JSON file.")
        return

    # --- 1. Export the Merged Version (WITH detailed brackets) ---
    df_merged["Supplier Segment"] = df_merged["Merged Account Types"].apply(lambda x: classify_supplier_mix(x, detailed=True))
    df_merged.to_csv(OUTPUT_MERGED_CSV, index=False)
    print(f"Exported merged data to: {OUTPUT_MERGED_CSV}")

    # --- 2. Create the Separated Columns Version (WITHOUT brackets) ---
    # First, dynamically get the dummy variables (the 1s and 0s)
    separated_columns = df_merged["Merged Account Types"].str.get_dummies(sep=", ")

    # We copy the base columns from the merged dataframe to construct the separated one
    base_columns = ["Supplier ID", "Supplier Name", "ERP Code", "Region", "Area", "Territory"]
    df_separated_base = df_merged[base_columns].copy()

    # Generate the clean "Mix" label without the brackets just for this sheet
    df_separated_base["Supplier Segment"] = df_merged["Merged Account Types"].apply(lambda x: classify_supplier_mix(x, detailed=False))

    # Combine the base columns + the segment column + the binary 1/0 columns
    df_separated = pd.concat([df_separated_base, separated_columns], axis=1)

    # Export it
    df_separated.to_csv(OUTPUT_SEPARATED_CSV, index=False)
    print(f"Exported separated columns data to: {OUTPUT_SEPARATED_CSV}")

    print("\nDone! All files successfully generated.")

if __name__ == "__main__":
    main()

Loading Target Supplier Data from 'distributor list.xlsx'...
Total Unique Supplier IDs extracted: 1424

Scanning 'sfa.accountCalls-Full.json' iteratively... (This may take a minute)

Processing matched data...
Exported merged data to: supplier_types_merged.csv
Exported separated columns data to: supplier_types_separated.csv

Done! All files successfully generated.
